# Practical 6 — Domain Shift: MegaDetector on Aerial Imagery

**Context:** MegaDetector was trained on millions of **camera trap** images —
side-view photos where animals are typically large, close, and at eye level.
What happens when we apply it to **aerial drone imagery** where animals are
tiny (30–60 px), viewed from above (nadir), and surrounded by savanna landscape?

This notebook uses the **Eikelboom et al. (2019)** dataset — aerial drone surveys
of elephants, zebras, and giraffes in Kenya — to quantify the domain shift.
We run MegaDetector v1000 (larch) on these images and compare its predictions
against the ground truth bounding boxes.

| Property | Camera trap (MD training data) | Eikelboom aerial |
|----------|-------------------------------|------------------|
| Viewpoint | Side-view, eye-level | Top-down (nadir) |
| Animal size | 100–1000+ px | 30–60 px |
| Image size | ~2 MP | ~14 MP (4603 x 3068) |
| Background | Forest floor, trails | Savanna, bare soil |
| Species appearance | Side profile | Top-down silhouette |

**Expected outcome:** MegaDetector will miss most animals — demonstrating
why domain-specific models (like the Eikelboom RetinaNet or HerdNet) exist.

**Reference:**
> Eikelboom et al. (2019). Improving the precision and accuracy of animal population
> estimates with aerial image object detection. *Methods in Ecology and Evolution*, 10(11).
> [DOI: 10.1111/2041-210X.13277](https://doi.org/10.1111/2041-210X.13277)

---

## Environment Setup

**Local (recommended):** use the `fit-training` conda environment:

```bash
conda env create -f environment-training.yml
conda activate fit-training
```

**Google Colab:** uncomment and run the cell below.

In [1]:
%matplotlib inline

In [2]:
# Colab only — install dependencies if not already available
# import sys

# !git clone  https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[training,dev]"
#
# # sys.path is handled by pip install -e above — no manual append needed

In [3]:
%matplotlib inline

from pathlib import Path
import json
import time

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image



# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")

from wildlife_detection.download_data import download_eikelboom
download_eikelboom(n_images=10, output_dir=DATA_DIR)

EIKELBOOM_DIR = DATA_DIR / "eikelboom"
TEST_IMG_DIR = EIKELBOOM_DIR / "test"
ANNOTATION_CSV = EIKELBOOM_DIR / "annotations" / "annotations_test.csv"
OUTPUT_DIR = DATA_DIR / "domain_shift_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TEST_IMG_DIR.exists(), f"Eikelboom test images not found at {TEST_IMG_DIR}"
assert ANNOTATION_CSV.exists(), f"Annotations not found at {ANNOTATION_CSV}"

print(f"Test images  : {TEST_IMG_DIR}")
print(f"Annotations  : {ANNOTATION_CSV}")
print(f"Output       : {OUTPUT_DIR}")


=== Eikelboom 2019 (n=10 per split) ===


ConnectError: [Errno 8] nodename nor servname provided, or not known

## 1 — Load ground truth

The Eikelboom annotations are CSV files with **pixel-level bounding boxes**
in Pascal VOC format: `filename, x1, y1, x2, y2, species`.

Three species: Elephant, Zebra, Giraffe — all viewed from above.

In [ ]:
gt = pd.read_csv(ANNOTATION_CSV, header=None,
                 names=["file", "x1", "y1", "x2", "y2", "species"])

# Strip path prefix — annotations have 'test/filename.JPG'
gt["filename"] = gt["file"].apply(lambda x: Path(x).name)
gt["w"] = gt["x2"] - gt["x1"]
gt["h"] = gt["y2"] - gt["y1"]

# Filter to images actually present on disk (subset may have been downloaded)
on_disk = {p.name for p in TEST_IMG_DIR.glob("*") if p.is_file()}
missing = set(gt["filename"].unique()) - on_disk
if missing:
    print(f"  Note: {len(missing)} annotation files not downloaded, skipping them.")
    gt = gt[gt["filename"].isin(on_disk)]

print(f"Ground truth: {len(gt)} annotations across {gt['filename'].nunique()} images")
print(f"\nSpecies counts:")
print(gt["species"].value_counts())
print(f"\nBox sizes (pixels):")
print(gt[["w", "h"]].describe().round(1))

In [ ]:
# Visualize a few images with ground truth
sample_files = gt["filename"].unique()[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, fname in zip(axes.flat, sample_files):
    img = Image.open(TEST_IMG_DIR / fname)
    ax.imshow(np.array(img))

    img_gt = gt[gt["filename"] == fname]
    colors = {"Elephant": "#E74C3C", "Zebra": "#3498DB", "Giraffe": "#F39C12"}
    for _, row in img_gt.iterrows():
        c = colors.get(row["species"], "white")
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1, edgecolor=c, facecolor="none",
        ))

    ax.set_title(f"{fname} — {len(img_gt)} animals ({img.size[0]}x{img.size[1]} px)", fontsize=9)
    ax.axis("off")

legend_patches = [mpatches.Patch(color=c, label=s) for s, c in colors.items()]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=10)
plt.suptitle("Eikelboom aerial dataset — ground truth", fontsize=13)
plt.tight_layout()

In [ ]:
# Zoom into individual animals to see what MD has to work with
# Pick one image with many annotations and crop around each GT box
dense_file = gt.groupby("filename").size().sort_values(ascending=False).index[0]
dense_gt = gt[gt["filename"] == dense_file]
full_img = np.array(Image.open(TEST_IMG_DIR / dense_file))

sample_rows = dense_gt.sample(min(8, len(dense_gt)), random_state=42)
n_show = len(sample_rows)
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
if n_show == 1:
    axes = [axes]

pad = 60  # pixels of context around each box
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    cx = (row["x1"] + row["x2"]) / 2
    cy = (row["y1"] + row["y2"]) / 2
    crop_x1 = max(0, int(cx - pad))
    crop_y1 = max(0, int(cy - pad))
    crop_x2 = min(full_img.shape[1], int(cx + pad))
    crop_y2 = min(full_img.shape[0], int(cy + pad))

    crop = full_img[crop_y1:crop_y2, crop_x1:crop_x2]
    ax.imshow(crop)

    # Draw GT box relative to crop origin
    ax.add_patch(mpatches.Rectangle(
        (row["x1"] - crop_x1, row["y1"] - crop_y1), row["w"], row["h"],
        linewidth=2, edgecolor="#2ECC71", facecolor="none",
    ))
    ax.set_title(f"{row['species']}\n{row['w']}x{row['h']} px", fontsize=8)
    ax.axis("off")

plt.suptitle(f"Zoomed ground truth animals from {dense_file}", fontsize=11)
plt.tight_layout()

Notice how small the animals are relative to the full image.
The average bounding box is ~50 x 58 pixels in a 4603 x 3068 image.
That's **~0.02%** of the image area per animal.

## 2 — Load MegaDetector v1000 (larch)

Same model as in the ultralytics notebook. MD v1000 larch is a YOLOv11-L
trained on camera trap images.

In [ ]:
import os
import torch
import wget
from ultralytics import YOLO

cache_dir = os.path.join(torch.hub.get_dir(), "checkpoints")
weights_path = os.path.join(cache_dir, "md_v1000.0.0-larch.pt")
if not os.path.exists(weights_path):
    os.makedirs(cache_dir, exist_ok=True)
    print("Downloading MD v1000 larch weights (~49 MB)...")
    wget.download(
        "https://github.com/agentmorris/MegaDetector/releases/download/v1000.0/md_v1000.0.0-larch.pt",
        out=weights_path,
    )
    print()

md_model = YOLO(weights_path)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# MD v1000 categories: 0=animal, 1=person, 2=vehicle
MD_LABELS = {0: "animal", 1: "person", 2: "vehicle"}

print(f"MegaDetector v1000 larch loaded")
print(f"Device: {DEVICE}")

## 3 — Run MegaDetector on Eikelboom test images

We run at the **default input resolution** first — the same way you would
run MegaDetector on camera trap images. No SAHI, no tiling, just the
standard `model.predict()` call.

The images are 4603 x 3068 but YOLO internally resizes to 640 px —
each animal shrinks to roughly **4–8 pixels**. Can the model still detect them?

Set `DEMO_MODE = True` to run on a small subset (5 images) for quick iteration.
Set it to `False` to run on the full test set (112 images) for proper evaluation.

In [ ]:
CONF_THRESHOLD = 0.3  # Low threshold to catch anything MD finds
DEMO_MODE = True      # True = 5 images (fast), False = full test set (112 images)

all_test_images = sorted(TEST_IMG_DIR.glob("*.JPG"))

if DEMO_MODE:
    # Pick 5 images with the most annotations for a representative demo
    gt_per_image = gt.groupby("filename").size().sort_values(ascending=False)
    demo_files = set(gt_per_image.head(5).index)
    test_images = [p for p in all_test_images if p.name in demo_files]
    print(f"DEMO MODE: running on {len(test_images)} of {len(all_test_images)} images")
    print(f"  (selected images with most annotations: {sorted(demo_files)})")
else:
    test_images = all_test_images
    print(f"Running MegaDetector on all {len(test_images)} aerial images...")

# Filter ground truth to match selected images
gt_filenames = {p.name for p in test_images}
gt_subset = gt[gt["filename"].isin(gt_filenames)]
print(f"  Ground truth for selected images: {len(gt_subset)} annotations")

md_results = []
t0 = time.time()

for img_path in test_images:
    results = md_model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
    boxes_obj = results[0].boxes

    dets = []
    for i in range(len(boxes_obj)):
        x1, y1, x2, y2 = boxes_obj.xyxy[i].tolist()
        cls_id = int(boxes_obj.cls[i])
        conf = float(boxes_obj.conf[i])
        dets.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "conf": conf, "class": cls_id,
            "label": MD_LABELS.get(cls_id, "unknown"),
        })

    md_results.append({"filename": img_path.name, "detections": dets})

elapsed = time.time() - t0

total_dets = sum(len(r["detections"]) for r in md_results)
animal_dets = sum(
    sum(1 for d in r["detections"] if d["class"] == 0)
    for r in md_results
)
gt_total = len(gt_subset)

print(f"\nDone in {elapsed:.1f}s")
print(f"Total detections (all classes): {total_dets}")
print(f"Animal detections: {animal_dets}")
print(f"Ground truth animals: {gt_total}")
print(f"\n→ MegaDetector found {animal_dets} of {gt_total} animals "
      f"({100 * animal_dets / gt_total:.1f}% raw recall)")

## 4 — Visual comparison: predictions vs. ground truth

Let's overlay both MegaDetector predictions (red dashed) and ground truth (green solid)
on the same images to see the domain shift visually.

In [ ]:
# Pick images with the most ground truth annotations
gt_counts = gt_subset.groupby("filename").size().sort_values(ascending=False)
show_files = gt_counts.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, fname in zip(axes.flat, show_files):
    img = np.array(Image.open(TEST_IMG_DIR / fname))
    ax.imshow(img)

    # Ground truth (green)
    img_gt = gt_subset[gt_subset["filename"] == fname]
    for _, row in img_gt.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1.2, edgecolor="#2ECC71", facecolor="none", linestyle="-",
        ))

    # MegaDetector predictions (red dashed)
    md_img = next((r for r in md_results if r["filename"] == fname), None)
    n_pred = 0
    if md_img:
        for det in md_img["detections"]:
            if det["class"] != 0:  # animals only
                continue
            ax.add_patch(mpatches.Rectangle(
                (det["x1"], det["y1"]),
                det["x2"] - det["x1"], det["y2"] - det["y1"],
                linewidth=1.5, edgecolor="#E74C3C", facecolor="none", linestyle="--",
            ))
            n_pred += 1

    ax.set_title(f"{fname} — GT: {len(img_gt)} | MD: {n_pred}", fontsize=9)
    ax.axis("off")

legend = [
    mpatches.Patch(edgecolor="#2ECC71", facecolor="none", label="Ground truth"),
    mpatches.Patch(edgecolor="#E74C3C", facecolor="none", label="MegaDetector", linestyle="--"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Domain shift: MegaDetector (camera trap model) on aerial imagery", fontsize=13)
plt.tight_layout()

Question: Why is Megadetector not finding any animals?

### Intersection over Union (IoU)

IoU measures how much two bounding boxes overlap:

```
         IoU = Intersection Area / Union Area
```

| IoU | Meaning |
|-----|---------|
| 1.0 | Boxes overlap perfectly |
| ≥ 0.5 | Standard match threshold (PASCAL VOC) |
| 0.0 | No overlap at all |

We use IoU to decide whether a prediction *matches* a ground truth box:
- **True Positive (TP)**: prediction with IoU ≥ threshold against an unmatched GT box
- **False Positive (FP)**: prediction with no matching GT box
- **False Negative (FN)**: GT box with no matching prediction


In [ ]:
def compute_iou(box_a, box_b):
    """
    Compute Intersection over Union between two boxes in [x1, y1, x2, y2] format.
    Returns a float in [0, 1]. Used to decide whether a prediction matches a GT box.
    """
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def match_detections(gt_df, predictions, iou_threshold=0.3):
    """
    Match predictions to ground truth boxes per image using greedy IoU matching.

    For each image, predictions are matched to GT boxes in order. A prediction
    is a TP if it has IoU >= iou_threshold with an unmatched GT box; otherwise FP.
    Unmatched GT boxes are FN. Returns a DataFrame with TP/FP/FN counts per image.
    """
    results = []

    all_filenames = set(gt_df["filename"].unique())
    for pred in predictions:
        all_filenames.add(pred["filename"])

    for fname in sorted(all_filenames):
        img_gt = gt_df[gt_df["filename"] == fname]
        gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()
        gt_matched = [False] * len(gt_boxes)

        pred_entry = next((r for r in predictions if r["filename"] == fname), None)
        pred_boxes = []
        if pred_entry:
            pred_boxes = [
                [d["x1"], d["y1"], d["x2"], d["y2"]]
                for d in pred_entry["detections"] if d["class"] == 0
            ]

        tp = 0
        fp = 0

        for pb in pred_boxes:
            best_iou = 0
            best_idx = -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j

            if best_iou >= iou_threshold:
                tp += 1
                gt_matched[best_idx] = True
            else:
                fp += 1

        fn = sum(1 for m in gt_matched if not m)

        results.append({
            "filename": fname,
            "gt_count": len(gt_boxes),
            "pred_count": len(pred_boxes),
            "tp": tp, "fp": fp, "fn": fn,
        })

    return pd.DataFrame(results)


In [ ]:
# IoU visualisation: three scenarios — perfect, partial, no overlap
import matplotlib.patches as patches

def draw_iou(ax, box_a, box_b, title):
    """Draw two boxes and shade their intersection."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect('equal')
    ax.invert_yaxis()

    # Union background
    ux1 = min(box_a[0], box_b[0]); uy1 = min(box_a[1], box_b[1])
    ux2 = max(box_a[2], box_b[2]); uy2 = max(box_a[3], box_b[3])
    ax.add_patch(patches.Rectangle((ux1, uy1), ux2-ux1, uy2-uy1,
                 linewidth=0, facecolor='#d0e8ff', label='Union'))

    # Intersection
    ix1 = max(box_a[0], box_b[0]); iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2]); iy2 = min(box_a[3], box_b[3])
    if ix2 > ix1 and iy2 > iy1:
        ax.add_patch(patches.Rectangle((ix1, iy1), ix2-ix1, iy2-iy1,
                     linewidth=0, facecolor='#6ab0ff', label='Intersection'))

    # GT box (green)
    ax.add_patch(patches.Rectangle((box_a[0], box_a[1]), box_a[2]-box_a[0], box_a[3]-box_a[1],
                 linewidth=2, edgecolor='green', facecolor='none', label='Ground truth'))
    # Pred box (red dashed)
    ax.add_patch(patches.Rectangle((box_b[0], box_b[1]), box_b[2]-box_b[0], box_b[3]-box_b[1],
                 linewidth=2, edgecolor='red', linestyle='--', facecolor='none', label='Prediction'))

    iou = compute_iou(box_a, box_b)
    ax.set_title(f"{title}\nIoU = {iou:.2f}", fontsize=10)
    ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

draw_iou(axes[0], [1, 1, 6, 6], [1, 1, 6, 6], "Perfect overlap")
draw_iou(axes[1], [1, 1, 6, 6], [3, 3, 8, 8], "Partial overlap")
draw_iou(axes[2], [1, 1, 4, 4], [6, 6, 9, 9], "No overlap")

# Shared legend on the middle plot
handles = [
    patches.Patch(facecolor='#6ab0ff', label='Intersection'),
    patches.Patch(facecolor='#d0e8ff', label='Union'),
    patches.Patch(edgecolor='green', facecolor='none', linewidth=2, label='Ground truth'),
    patches.Patch(edgecolor='red', facecolor='none', linewidth=2, linestyle='--', label='Prediction'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9, frameon=False)
plt.suptitle("Intersection over Union (IoU) — three scenarios", fontsize=12)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


## 4b — Can SAHI (tiled inference) rescue MegaDetector?

SAHI splits large images into overlapping tiles, runs detection on each tile,
then merges results with NMS. This should help because each tile is closer to
camera-trap scale — the animals become larger relative to the input.

We use 640x640 tiles with 20% overlap.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from tqdm.auto import tqdm

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=weights_path,
    confidence_threshold=CONF_THRESHOLD,
    device=DEVICE,
)

SLICE_SIZE = 640 # size of a tile
OVERLAP_RATIO = 0.2 # how much the tiles overlap

print(f"Running SAHI tiled inference on {len(test_images)} images...")
print(f"  Tile size: {SLICE_SIZE}x{SLICE_SIZE}, overlap: {OVERLAP_RATIO:.0%}")
print(f"  (~56 tiles per 4603x3068 image)\n")

sahi_results = []
t0 = time.time()

for img_path in tqdm(test_images, desc="SAHI inference"):
    result = get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )

    dets = []
    for pred in result.object_prediction_list:
        bbox = pred.bbox
        cls_id = pred.category.id
        conf = pred.score.value
        dets.append({
            "x1": bbox.minx, "y1": bbox.miny,
            "x2": bbox.maxx, "y2": bbox.maxy,
            "conf": conf, "class": cls_id,
            "label": MD_LABELS.get(cls_id, "unknown"),
        })

    sahi_results.append({"filename": img_path.name, "detections": dets})

sahi_elapsed = time.time() - t0

sahi_animal_dets = sum(
    sum(1 for d in r["detections"] if d["class"] == 0)
    for r in sahi_results
)

print(f"\nDone in {sahi_elapsed:.1f}s (vs {elapsed:.1f}s without SAHI)")
print(f"SAHI animal detections: {sahi_animal_dets}")
print(f"Standard MD detections: {animal_dets}")
print(f"Ground truth animals:   {len(gt)}")

In [ ]:
# Visual comparison: Standard MD vs SAHI on the densest images
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
compare_files = gt_counts.head(4).index.tolist()

for ax, fname in zip(axes.flat, compare_files):
    img_arr = np.array(Image.open(TEST_IMG_DIR / fname))
    ax.imshow(img_arr)

    # Ground truth (green)
    img_gt = gt[gt["filename"] == fname]
    for _, row in img_gt.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1, edgecolor="#2ECC71", facecolor="none",
        ))

    # SAHI predictions (orange dashed)
    sahi_img = next((r for r in sahi_results if r["filename"] == fname), None)
    n_sahi = 0
    if sahi_img:
        for det in sahi_img["detections"]:
            if det["class"] != 0:
                continue
            ax.add_patch(mpatches.Rectangle(
                (det["x1"], det["y1"]),
                det["x2"] - det["x1"], det["y2"] - det["y1"],
                linewidth=1.5, edgecolor="#F39C12", facecolor="none", linestyle="--",
            ))
            n_sahi += 1

    n_gt = len(img_gt)
    ax.set_title(f"{fname} — GT: {n_gt} | SAHI: {n_sahi}", fontsize=9)
    ax.axis("off")

legend = [
    mpatches.Patch(edgecolor="#2ECC71", facecolor="none", label="Ground truth"),
    mpatches.Patch(edgecolor="#F39C12", facecolor="none", label="MD + SAHI", linestyle="--"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("SAHI tiled inference: does slicing help MegaDetector on aerial images?", fontsize=13)
plt.tight_layout()

In [ ]:
# Zoom into SAHI detections — what is MD actually detecting from above?
# Pick the image with the most SAHI animal detections
sahi_counts = {r["filename"]: sum(1 for d in r["detections"] if d["class"] == 0) for r in sahi_results}
sahi_dense_file = max(sahi_counts, key=sahi_counts.get)
sahi_dense = next(r for r in sahi_results if r["filename"] == sahi_dense_file)
sahi_dense_img = np.array(Image.open(TEST_IMG_DIR / sahi_dense_file))

animal_dets_list = [d for d in sahi_dense["detections"] if d["class"] == 0]
sample_dets = animal_dets_list[:8]  # first 8 detections
n_show = len(sample_dets)

if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    pad = 60
    for ax, det in zip(axes, sample_dets):
        cx = (det["x1"] + det["x2"]) / 2
        cy = (det["y1"] + det["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(sahi_dense_img.shape[1], int(cx + pad))
        crop_y2 = min(sahi_dense_img.shape[0], int(cy + pad))

        crop = sahi_dense_img[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        # Draw SAHI detection box relative to crop
        bw = det["x2"] - det["x1"]
        bh = det["y2"] - det["y1"]
        ax.add_patch(mpatches.Rectangle(
            (det["x1"] - crop_x1, det["y1"] - crop_y1), bw, bh,
            linewidth=2, edgecolor="#F39C12", facecolor="none", linestyle="--",
        ))
        ax.set_title(f"{det['label']} {det['conf']:.2f}\n{bw:.0f}x{bh:.0f} px", fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Zoomed SAHI detections from {sahi_dense_file}", fontsize=11)
    plt.tight_layout()
else:
    print(f"No SAHI animal detections to display.")

In [ ]:
# Zoom into ONLY high-confidence SAHI detections (conf >= 0.5)
# These are what the model is "most sure" about — are they real animals or false positives?
HIGH_CONF = 0.5

high_conf_dets = []
for r in sahi_results:
    for d in r["detections"]:
        if d["class"] == 0 and d["conf"] >= HIGH_CONF:
            high_conf_dets.append({**d, "filename": r["filename"]})

print(f"High-confidence SAHI detections (conf >= {HIGH_CONF}): {len(high_conf_dets)}")

n_show = min(8, len(high_conf_dets))
if n_show > 0:
    # Sort by confidence descending
    high_conf_dets.sort(key=lambda d: d["conf"], reverse=True)
    show_dets = high_conf_dets[:n_show]

    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    pad = 60
    for ax, det in zip(axes, show_dets):
        img_arr = np.array(Image.open(TEST_IMG_DIR / det["filename"]))
        cx = (det["x1"] + det["x2"]) / 2
        cy = (det["y1"] + det["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(img_arr.shape[1], int(cx + pad))
        crop_y2 = min(img_arr.shape[0], int(cy + pad))

        crop = img_arr[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        bw = det["x2"] - det["x1"]
        bh = det["y2"] - det["y1"]
        ax.add_patch(mpatches.Rectangle(
            (det["x1"] - crop_x1, det["y1"] - crop_y1), bw, bh,
            linewidth=2, edgecolor="#E74C3C", facecolor="none",
        ))
        ax.set_title(f"conf={det['conf']:.2f}\n{bw:.0f}x{bh:.0f} px", fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Highest-confidence SAHI detections (conf >= {HIGH_CONF}) — real animals or false positives?", fontsize=11)
    plt.tight_layout()
else:
    print(f"No detections above conf {HIGH_CONF}. Try lowering the threshold.")

Question: What is the difference between high confidence and low confidence predictions?


In [ ]:
# False negatives: animals that NEITHER standard MD nor SAHI detected
# These illustrate the core domain shift problem but also the difference between the camera trap images and the aerial images.

def get_detected_gt_indices(gt_df, predictions, iou_threshold=0.3):
    """Return set of GT row indices that were matched by at least one prediction."""
    matched = set()
    for fname in gt_df["filename"].unique():
        img_gt = gt_df[gt_df["filename"] == fname]
        gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()
        gt_indices = img_gt.index.tolist()

        pred_entry = next((r for r in predictions if r["filename"] == fname), None)
        if not pred_entry:
            continue
        pred_boxes = [[d["x1"], d["y1"], d["x2"], d["y2"]]
                      for d in pred_entry["detections"] if d["class"] == 0]

        for pb in pred_boxes:
            best_iou, best_idx = 0, -1
            for j, gb in enumerate(gt_boxes):
                iou = compute_iou(pb, gb)
                if iou > best_iou and gt_indices[j] not in matched:
                    best_iou = iou
                    best_idx = gt_indices[j]
            if best_iou >= iou_threshold and best_idx >= 0:
                matched.add(best_idx)
    return matched

# Find GT annotations missed by both standard MD and SAHI
md_detected = get_detected_gt_indices(gt_subset, md_results)
sahi_detected = get_detected_gt_indices(gt_subset, sahi_results)
all_detected = md_detected | sahi_detected
missed_mask = ~gt_subset.index.isin(all_detected)
missed = gt_subset[missed_mask]

print(f"Ground truth animals:        {len(gt_subset)}")
print(f"Detected by standard MD:     {len(md_detected)}")
print(f"Detected by SAHI:            {len(sahi_detected)}")
print(f"Missed by BOTH:              {len(missed)}")
print(f"→ {100 * len(missed) / len(gt_subset):.0f}% of animals are invisible to MegaDetector")
print(f"\nMissed by species:")
print(missed["species"].value_counts())

In [ ]:
# Visualize false negatives — zoomed crops of animals both MD and SAHI missed
# Since MD barely detects anything, most GT annotations are false negatives.
# Sample a diverse set across images for visual impact.

if len(missed) == 0:
    print("No false negatives — all animals were detected (unexpected!).")
else:
    # Sample up to 9 missed animals, spread across different images
    missed_shuffled = missed.sample(frac=1, random_state=42)
    missed_sample = missed_shuffled.drop_duplicates("filename").head(9)
    # If fewer than 9 images, fill with more from the same images
    if len(missed_sample) < 9:
        remaining = missed_shuffled[~missed_shuffled.index.isin(missed_sample.index)]
        missed_sample = pd.concat([missed_sample, remaining.head(9 - len(missed_sample))])

    n_show = min(9, len(missed_sample))
    cols = 3
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    pad = 80
    for idx, (ax, (_, row)) in enumerate(zip(axes_flat, missed_sample.head(n_show).iterrows())):
        img_arr = np.array(Image.open(TEST_IMG_DIR / row["filename"]))
        cx = (row["x1"] + row["x2"]) / 2
        cy = (row["y1"] + row["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(img_arr.shape[1], int(cx + pad))
        crop_y2 = min(img_arr.shape[0], int(cy + pad))

        crop = img_arr[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        # Draw GT box (green) — the animal that was missed
        ax.add_patch(mpatches.Rectangle(
            (row["x1"] - crop_x1, row["y1"] - crop_y1), row["w"], row["h"],
            linewidth=2, edgecolor="#2ECC71", facecolor="none",
        ))
        species = row.get("species", "unknown")
        ax.set_title(f"{species} — MISSED\n{row['w']}x{row['h']} px", fontsize=9)
        ax.axis("off")

    # Hide unused subplots
    for j in range(n_show, rows * cols):
        axes_flat[j].axis("off") if j < len(list(axes_flat)) else None

    plt.suptitle("False negatives: animals that MegaDetector + SAHI both missed", fontsize=12)
    plt.tight_layout()

Calculation of some Metrics like Recall, Precision and F1 Score

In [ ]:
# Compare metrics: Standard MD vs SAHI
# Compute standard MD metrics here so this cell is self-contained
md_metrics_df = match_detections(gt_subset, md_results, iou_threshold=0.3)
md_TP = md_metrics_df["tp"].sum()
md_FP = md_metrics_df["fp"].sum()
md_FN = md_metrics_df["fn"].sum()
md_precision = md_TP / (md_TP + md_FP) if (md_TP + md_FP) > 0 else 0
md_recall = md_TP / (md_TP + md_FN) if (md_TP + md_FN) > 0 else 0
md_f1 = 2 * md_precision * md_recall / (md_precision + md_recall) if (md_precision + md_recall) > 0 else 0
md_mae = (md_metrics_df["pred_count"] - md_metrics_df["gt_count"]).abs().mean()
md_total_pred = md_metrics_df["pred_count"].sum()

sahi_metrics_df = match_detections(gt_subset, sahi_results, iou_threshold=0.3)
sahi_TP = sahi_metrics_df["tp"].sum()
sahi_FP = sahi_metrics_df["fp"].sum()
sahi_FN = sahi_metrics_df["fn"].sum()
sahi_precision = sahi_TP / (sahi_TP + sahi_FP) if (sahi_TP + sahi_FP) > 0 else 0
sahi_recall = sahi_TP / (sahi_TP + sahi_FN) if (sahi_TP + sahi_FN) > 0 else 0
sahi_f1 = 2 * sahi_precision * sahi_recall / (sahi_precision + sahi_recall) if (sahi_precision + sahi_recall) > 0 else 0
sahi_mae = (sahi_metrics_df["pred_count"] - sahi_metrics_df["gt_count"]).abs().mean()
sahi_total_pred = sahi_metrics_df["pred_count"].sum()

comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "MAE", "Total detections", "TP", "FP", "FN"],
    "Standard MD": [f"{md_precision:.3f}", f"{md_recall:.3f}", f"{md_f1:.3f}", f"{md_mae:.2f}",
                    str(md_total_pred), str(md_TP), str(md_FP), str(md_FN)],
    "MD + SAHI": [f"{sahi_precision:.3f}", f"{sahi_recall:.3f}", f"{sahi_f1:.3f}", f"{sahi_mae:.2f}",
                  str(sahi_total_pred), str(sahi_TP), str(sahi_FP), str(sahi_FN)],
})

print("=" * 60)
print(f"  {'Metric':<20s}  {'Standard MD':>12s}  {'MD + SAHI':>12s}")
print("=" * 60)
for _, row in comparison.iterrows():
    print(f"  {row['Metric']:<20s}  {row['Standard MD']:>12s}  {row['MD + SAHI']:>12s}")
print("=" * 60)
print(f"\n  Ground truth: {gt_total} animals")
print(f"  SAHI tile size: {SLICE_SIZE}x{SLICE_SIZE}, overlap: {OVERLAP_RATIO:.0%}")
print(f"  Time: standard {elapsed:.1f}s vs SAHI {sahi_elapsed:.1f}s")

In [ ]:
# Confidence distribution of SAHI detections vs. ground truth overlap
# For each SAHI detection, compute its best IoU with any GT box

det_confs = []  # (conf, best_iou, is_tp)
for r in sahi_results:
    fname = r["filename"]
    img_gt = gt_subset[gt_subset["filename"] == fname]
    gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()

    for d in r["detections"]:
        if d["class"] != 0:
            continue
        pred_box = [d["x1"], d["y1"], d["x2"], d["y2"]]
        best_iou = max((compute_iou(pred_box, gb) for gb in gt_boxes), default=0.0)
        det_confs.append({
            "conf": d["conf"],
            "best_iou": best_iou,
            "tp": best_iou >= 0.3,
        })

det_df = pd.DataFrame(det_confs)

if len(det_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: confidence histogram, colored by TP/FP
    ax = axes[0]
    tp_confs = det_df[det_df["tp"]]["conf"]
    fp_confs = det_df[~det_df["tp"]]["conf"]
    ax.hist([tp_confs, fp_confs], bins=20, stacked=True,
            color=["#2ECC71", "#E74C3C"], label=["True Positive", "False Positive"],
            edgecolor="white")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Number of detections")
    ax.set_title("SAHI detection confidence distribution")
    ax.legend()

    # Right: confidence vs IoU scatter
    ax = axes[1]
    colors = ["#2ECC71" if tp else "#E74C3C" for tp in det_df["tp"]]
    ax.scatter(det_df["conf"], det_df["best_iou"], c=colors, alpha=0.5, s=20)
    ax.axhline(0.3, color="gray", linestyle="--", alpha=0.5, label="IoU threshold")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Best IoU with GT")
    ax.set_title("Confidence vs. IoU overlap")
    ax.legend()

    plt.suptitle("Are confident detections correct? (SAHI results)", fontsize=12)
    plt.tight_layout()
else:
    print("No SAHI animal detections to analyze.")

In [ ]:
# Precision-Recall curve by sweeping confidence threshold
# For each threshold, compute P/R from SAHI detections vs GT

thresholds = np.arange(0.05, 0.95, 0.02)
pr_points = []

for t in thresholds:
    filtered = []
    for r in sahi_results:
        filtered.append({
            "filename": r["filename"],
            "detections": [d for d in r["detections"] if d["conf"] >= t],
        })

    mdf = match_detections(gt_subset, filtered, iou_threshold=0.3)
    tp = mdf["tp"].sum()
    fp = mdf["fp"].sum()
    fn = mdf["fn"].sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 1.0  # no predictions = perfect precision
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pr_points.append({"threshold": t, "precision": p, "recall": r})

pr_df = pd.DataFrame(pr_points)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision-Recall curve
ax = axes[0]
ax.plot(pr_df["recall"], pr_df["precision"], "o-", color="steelblue", markersize=3)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curve (SAHI + MegaDetector)")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# Annotate a few threshold values
for t_label in [0.1, 0.3, 0.5, 0.7]:
    row = pr_df.iloc[(pr_df["threshold"] - t_label).abs().argmin()]
    ax.annotate(f"t={t_label}", (row["recall"], row["precision"]),
                fontsize=7, color="red", textcoords="offset points", xytext=(5, 5))

# Right: P, R, F1 vs threshold
ax = axes[1]
pr_df["f1"] = 2 * pr_df["precision"] * pr_df["recall"] / (pr_df["precision"] + pr_df["recall"]).replace(0, 1e-8)
ax.plot(pr_df["threshold"], pr_df["precision"], label="Precision", color="#E74C3C")
ax.plot(pr_df["threshold"], pr_df["recall"], label="Recall", color="#3498DB")
ax.plot(pr_df["threshold"], pr_df["f1"], label="F1", color="#2ECC71", linewidth=2)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Score")
ax.set_title("P / R / F1 vs. confidence threshold")
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# Find best F1
best_row = pr_df.loc[pr_df["f1"].idxmax()]
ax.axvline(best_row["threshold"], color="gray", linestyle="--", alpha=0.5)
ax.annotate(f"best F1={best_row['f1']:.3f}\nt={best_row['threshold']:.2f}",
            (best_row["threshold"], best_row["f1"]),
            fontsize=8, textcoords="offset points", xytext=(10, -15))

plt.suptitle("MegaDetector + SAHI on aerial data: no threshold saves the model", fontsize=12)
plt.tight_layout()

## 5 — The domain shift verdict

MegaDetector — even with SAHI tiled inference — cannot detect all animals and finds many false positives
in aerial imagery. The model was trained on side-view camera trap
photos where animals are large, centred, and at eye level. From above,
the same animals are tiny, top-down silhouettes against a uniform background.

**This is not a threshold problem or a resolution problem — it is a
fundamental domain mismatch.** The solution is to train a domain-specific
model on aerial data (see the training practical).

## 5b — Can fine-tuning bridge the domain gap?

MegaDetector fails because it was trained on camera trap images. What if we
**fine-tune** it on aerial wildlife data?

**AerialDetector** is a YOLO11-L model initialised from MegaDetector v1000 (larch)
and fine-tuned on ~28,000 aerial wildlife tiles from four datasets:

| Dataset | Images | Species | Source |
|---------|--------|---------|--------|
| Eikelboom (2019) | 1,968 tiles | Elephant, Giraffe, Zebra | Oblique aircraft, Kenya |
| Koger et al. (2023) | 55,064 tiles | Zebra, Gazelle, Waterbuck, Buffalo, Gelada | Nadir drone, Kenya + Ethiopia |
| Delplanque et al. (2022) | 86,968 tiles | 6 African savanna species | Nadir UAV, 4 countries |
| WAID (Mou et al., 2023) | 10,056 tiles | Zebra, Cattle, Camel, Kiang, Sheep, Seal | Nadir + oblique drone |

All species mapped to a single **"animal"** class (MegaDetector convention).
Training used `freeze=10` (backbone frozen, detection head only), 50 epochs.

Published at [huggingface.co/karisu/aerialdetector](https://huggingface.co/karisu/aerialdetector).



---

### How was AerialDetector trained?

Fine-tuning starts from MegaDetector v1000 (larch) weights and updates only the
detection head for the first 50 epochs (`freeze=10` — backbone frozen).
All species are mapped to a single `animal` class.

You can reproduce this with the training script in this repo:

```bash
python scripts/training/train_combined_yolo11.py \
    --weights weights/md_v1000.0.0-larch.pt \
    --data week1/data/eikelboom_yolo_tiled/dataset.yaml \
    --epochs 50 --freeze 10 --name aerialdetector-repro
```

For deeper domain adaptation, `phased_finetune.py` does 3-phase progressive
unfreezing (head only → partial → full backbone). See `doc/fine_tuning_yolo11.md`
for the full strategy and dataset analysis.


In [ ]:
from huggingface_hub import hf_hub_download

# Download AerialDetector weights from HuggingFace
aerial_weights = hf_hub_download(
    repo_id="karisu/aerialdetector",
    filename="larch-freeze10-combined_waid/weights/best.pt",
)
print(f"AerialDetector weights: {aerial_weights}")

aerial_model = YOLO(aerial_weights)
print(f"Classes: {aerial_model.names}")
print(f"Parameters: {sum(p.numel() for p in aerial_model.parameters()):,}")


## 5c — AerialDetector + SAHI on Eikelboom test images

Same SAHI setup as before (640x640 tiles, 20% overlap), but now with a model
that was actually trained on aerial imagery.


In [ ]:
aerial_sahi_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=aerial_weights,
    confidence_threshold=CONF_THRESHOLD,
    device=DEVICE,
)

print(f"Running AerialDetector + SAHI on {len(test_images)} images...")
print(f"  Tile size: {SLICE_SIZE}x{SLICE_SIZE}, overlap: {OVERLAP_RATIO:.0%}")

aerial_sahi_results = []
t0 = time.time()

for img_path in tqdm(test_images, desc="AerialDetector + SAHI"):
    result = get_sliced_prediction(
        str(img_path),
        aerial_sahi_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )

    dets = []
    for pred in result.object_prediction_list:
        bbox = pred.bbox
        dets.append({
            "x1": bbox.minx, "y1": bbox.miny,
            "x2": bbox.maxx, "y2": bbox.maxy,
            "conf": pred.score.value,
            "class": pred.category.id,
            "label": "animal",
        })
    aerial_sahi_results.append({"filename": img_path.name, "detections": dets})

aerial_elapsed = time.time() - t0
aerial_animal_dets = sum(len(r["detections"]) for r in aerial_sahi_results)

print(f"\nDone in {aerial_elapsed:.1f}s")
print(f"AerialDetector detections: {aerial_animal_dets}")
print(f"MegaDetector SAHI dets:    {sahi_animal_dets}")
print(f"Ground truth animals:      {len(gt_subset)}")


In [ ]:
# Visual comparison: GT vs AerialDetector SAHI on the densest images
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
compare_files = gt_counts.head(4).index.tolist()

for ax, fname in zip(axes.flat, compare_files):
    img_arr = np.array(Image.open(TEST_IMG_DIR / fname))
    ax.imshow(img_arr)

    # Ground truth (green)
    img_gt = gt[gt["filename"] == fname]
    for _, row in img_gt.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1, edgecolor="#2ECC71", facecolor="none",
        ))

    # AerialDetector SAHI predictions (cyan dashed)
    aerial_img = next((r for r in aerial_sahi_results if r["filename"] == fname), None)
    n_aerial = 0
    if aerial_img:
        for det in aerial_img["detections"]:
            ax.add_patch(mpatches.Rectangle(
                (det["x1"], det["y1"]),
                det["x2"] - det["x1"], det["y2"] - det["y1"],
                linewidth=1.5, edgecolor="#9B59B6", facecolor="none", linestyle="--",
            ))
            n_aerial += 1

    n_gt = len(img_gt)
    ax.set_title(f"{fname} \u2014 GT: {n_gt} | Aerial: {n_aerial}", fontsize=9)
    ax.axis("off")

legend = [
    mpatches.Patch(edgecolor="#2ECC71", facecolor="none", label="Ground truth"),
    mpatches.Patch(edgecolor="#9B59B6", facecolor="none", label="AerialDetector + SAHI", linestyle="--"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("AerialDetector + SAHI: does fine-tuning bridge the domain gap?", fontsize=13)
plt.tight_layout()


In [ ]:
# Zoom into AerialDetector detections — what is it finding?
aerial_counts = {r["filename"]: len(r["detections"]) for r in aerial_sahi_results}
aerial_dense_file = max(aerial_counts, key=aerial_counts.get)
aerial_dense = next(r for r in aerial_sahi_results if r["filename"] == aerial_dense_file)
aerial_dense_img = np.array(Image.open(TEST_IMG_DIR / aerial_dense_file))

sample_dets = aerial_dense["detections"][:8]
n_show = len(sample_dets)

if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    pad = 60
    for ax, det in zip(axes, sample_dets):
        cx = (det["x1"] + det["x2"]) / 2
        cy = (det["y1"] + det["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(aerial_dense_img.shape[1], int(cx + pad))
        crop_y2 = min(aerial_dense_img.shape[0], int(cy + pad))

        crop = aerial_dense_img[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        bw = det["x2"] - det["x1"]
        bh = det["y2"] - det["y1"]
        ax.add_patch(mpatches.Rectangle(
            (det["x1"] - crop_x1, det["y1"] - crop_y1), bw, bh,
            linewidth=2, edgecolor="#9B59B6", facecolor="none", linestyle="--",
        ))
        ax.set_title(f"{det['label']} {det['conf']:.2f}\n{bw:.0f}x{bh:.0f} px", fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Zoomed AerialDetector detections from {aerial_dense_file}", fontsize=11)
    plt.tight_layout()
else:
    print("No AerialDetector detections to display.")


In [ ]:
# False negatives: animals that AerialDetector + SAHI missed
aerial_detected = get_detected_gt_indices(gt_subset, aerial_sahi_results)

aerial_missed_mask = ~gt_subset.index.isin(aerial_detected)
aerial_missed = gt_subset[aerial_missed_mask]

print(f"Ground truth animals:            {len(gt_subset)}")
print(f"Detected by MD + SAHI:           {len(sahi_detected)}")
print(f"Detected by AerialDetector+SAHI: {len(aerial_detected)}")
print(f"Missed by AerialDetector:        {len(aerial_missed)}")
print(f"\u2192 {100 * len(aerial_missed) / len(gt_subset):.0f}% of animals missed (vs {100 * len(missed) / len(gt_subset):.0f}% for MegaDetector)")
print(f"\nMissed by species:")
print(aerial_missed["species"].value_counts())


In [ ]:
# Zoomed crops of animals AerialDetector missed
if len(aerial_missed) == 0:
    print("No false negatives \u2014 all animals were detected!")
else:
    missed_shuffled = aerial_missed.sample(frac=1, random_state=42)
    missed_sample = missed_shuffled.drop_duplicates("filename").head(9)
    if len(missed_sample) < 9:
        remaining = missed_shuffled[~missed_shuffled.index.isin(missed_sample.index)]
        missed_sample = pd.concat([missed_sample, remaining.head(9 - len(missed_sample))])

    n_show = min(9, len(missed_sample))
    cols = 3
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    pad = 80
    for idx, (ax, (_, row)) in enumerate(zip(axes_flat, missed_sample.head(n_show).iterrows())):
        img_arr = np.array(Image.open(TEST_IMG_DIR / row["filename"]))
        cx = (row["x1"] + row["x2"]) / 2
        cy = (row["y1"] + row["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(img_arr.shape[1], int(cx + pad))
        crop_y2 = min(img_arr.shape[0], int(cy + pad))

        crop = img_arr[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        ax.add_patch(mpatches.Rectangle(
            (row["x1"] - crop_x1, row["y1"] - crop_y1), row["w"], row["h"],
            linewidth=2, edgecolor="#2ECC71", facecolor="none",
        ))
        species = row.get("species", "unknown")
        ax.set_title(f"{species} \u2014 MISSED\n{row['w']}x{row['h']} px", fontsize=9)
        ax.axis("off")

    for j in range(n_show, rows * cols):
        axes_flat[j].axis("off") if j < len(list(axes_flat)) else None

    plt.suptitle("False negatives: animals that AerialDetector + SAHI missed", fontsize=12)
    plt.tight_layout()


In [ ]:
# Compare all three approaches
aerial_metrics_df = match_detections(gt_subset, aerial_sahi_results, iou_threshold=0.3)
aerial_TP    = aerial_metrics_df["tp"].sum()
aerial_FP    = aerial_metrics_df["fp"].sum()
aerial_FN    = aerial_metrics_df["fn"].sum()
aerial_precision = aerial_TP / (aerial_TP + aerial_FP) if (aerial_TP + aerial_FP) > 0 else 0
aerial_recall    = aerial_TP / (aerial_TP + aerial_FN) if (aerial_TP + aerial_FN) > 0 else 0
aerial_f1        = 2 * aerial_precision * aerial_recall / (aerial_precision + aerial_recall) if (aerial_precision + aerial_recall) > 0 else 0
aerial_total_pred = aerial_metrics_df["pred_count"].sum()

# Counting errors: MAE (magnitude) and ME (signed — positive = over-counting)
count_err = aerial_metrics_df["pred_count"] - aerial_metrics_df["gt_count"]
aerial_mae = count_err.abs().mean()
aerial_me  = count_err.mean()

# Pull matching values from the earlier cell (already computed)
md_me   = (md_metrics_df["pred_count"]   - md_metrics_df["gt_count"]).mean()
sahi_me = (sahi_metrics_df["pred_count"] - sahi_metrics_df["gt_count"]).mean()

print("=" * 75)
print(f"  {'Metric':<20s}  {'Standard MD':>12s}  {'MD + SAHI':>12s}  {'Aerial+SAHI':>12s}")
print("=" * 75)
rows = [
    ("Precision",      md_precision,   sahi_precision,   aerial_precision),
    ("Recall",         md_recall,      sahi_recall,      aerial_recall),
    ("F1",             md_f1,          sahi_f1,          aerial_f1),
    ("─" * 20,         "─" * 12,       "─" * 12,         "─" * 12),
    ("MAE (counts)",   md_mae,         sahi_mae,         aerial_mae),
    ("ME  (counts)",   md_me,          sahi_me,          aerial_me),
    ("─" * 20,         "─" * 12,       "─" * 12,         "─" * 12),
    ("Total dets",     md_total_pred,  sahi_total_pred,  aerial_total_pred),
    ("TP",             md_TP,          sahi_TP,          aerial_TP),
    ("FP",             md_FP,          sahi_FP,          aerial_FP),
    ("FN",             md_FN,          sahi_FN,          aerial_FN),
]
for name, v1, v2, v3 in rows:
    if isinstance(v1, float):
        print(f"  {name:<20s}  {v1:>12.3f}  {v2:>12.3f}  {v3:>12.3f}")
    elif isinstance(v1, str) and v1.startswith('─'):
        print(f"  {name}  {v1}  {v2}  {v3}")
    else:
        print(f"  {name:<20s}  {v1:>12}  {v2:>12}  {v3:>12}")
print("=" * 75)
print(f"\n  Ground truth: {gt_total} animals")
print(f"  MAE = Mean Absolute Error per image  |pred - gt|")
print(f"  ME  = Mean Error per image  pred - gt  (+ over-counts, − under-counts)")


## Exercises

1. the ultralytics YOLO is somewhat flexible with image input sizes. You don't need to change the training tile size (640x640) to run inference on larger images. Try running MegaDetector with `imgsz=1280` (instead of 640). Does it improve detection? Why or why not? What else do you observe?

1. **Confidence threshold tuning** — The AerialDetector comparison used conf=0.3.
   Sweep from 0.1 to 0.9 and plot a PR curve for AerialDetector + SAHI.
   What is the optimal threshold? How does the curve compare to MegaDetector's?

2. **SAHI tile size** — We used 640x640 tiles. Try 512x512 and 1024x1024
   with AerialDetector. Does tile size matter more or less now that the model
   was trained on 640x640 tiles?

3. **Resolution experiment** — Run AerialDetector without SAHI at `imgsz=640`
   and `imgsz=1280`. How much does SAHI help when the model already knows
   what aerial animals look like?

4. **False positive analysis** — Zoom into AerialDetector's false positives.
   Are they different from MegaDetector's? Does fine-tuning reduce the
   "detecting trees and shadows" problem?

5. **Compare with Eikelboom's RetinaNet** — The paper reports 90–95% recall.
   How close does AerialDetector get? What might explain any remaining gap?
   (Hint: Eikelboom trained on the exact same dataset; AerialDetector trained
   on a mix of datasets.)


## Reflection

- MegaDetector fails on aerial imagery but AerialDetector succeeds — both are
  YOLO11-L with 25M parameters. What changed? Only the training data.
  What does this tell you about the importance of training data vs. architecture?

- AerialDetector was fine-tuned with `freeze=10` (backbone frozen). This means
  the low-level features (edges, textures) from MegaDetector's camera-trap
  training were kept, but the detection head was retrained. Why does this work?

- The Eikelboom images are oblique (side-facing from aircraft), while most
  training data is nadir (top-down from drones). Despite this geometry mismatch,
  AerialDetector works. What does this suggest about the generalisability of
  aerial fine-tuning?

- If you were starting a new aerial wildlife survey, would you:
  (a) use MegaDetector as-is,
  (b) fine-tune MegaDetector on aerial data (like AerialDetector),
  (c) train from scratch on your specific survey data,
  (d) use a point-based detector like HerdNet?
  What factors would influence your decision?
